# MCTNet end-to-end from Google Drive

Ce notebook execute toute la chaine :
1. montage de Google Drive
2. lecture des CSV exportes depuis Google Earth Engine
3. construction des datasets `.npz/.json`
4. validation des dimensions
5. entrainement de MCTNet
6. matrice de confusion sur le jeu de test


In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive monte.')
except ImportError:
    print('Google Colab non detecte. Adapte les chemins si tu es en local.')


In [ ]:
# -----------------------------
# Configuration
# -----------------------------

# Dossier contenant les fichiers Python du projet.
PROJECT_DIR = Path('/content/drive/MyDrive/New project/python')

# Dossier contenant les CSV exportes depuis GEE.
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/mctnet_crop_mapping_2021')

# CSV bruts attendus.
CSV_FILES = [
    DRIVE_BASE_DIR / 'mctnet_samples_AR_2021.csv',
    DRIVE_BASE_DIR / 'mctnet_samples_CA_2021.csv',
]

# Dossier des datasets preprocesses.
PROCESSED_DIR = DRIVE_BASE_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Etat a entrainer: 'arkansas' ou 'california'.
STATE_TO_TRAIN = 'arkansas'

# Dossier de sortie de l'entrainement.
RUN_DIR = PROCESSED_DIR / 'training_runs' / f'{STATE_TO_TRAIN}_mctnet_run_01'
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparametres principaux.
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
EARLY_STOPPING_PATIENCE = 15
SEED = 2021

sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('DRIVE_BASE_DIR =', DRIVE_BASE_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)
print('RUN_DIR =', RUN_DIR)
print('STATE_TO_TRAIN =', STATE_TO_TRAIN)

for csv_path in CSV_FILES:
    print(csv_path, 'exists =', csv_path.exists())


In [ ]:
from build_mctnet_dataset import build_dataset_bundle, save_bundle
from validate_mctnet_tensors import load_bundle as load_npz_bundle, validate_split_shapes, validate_dataloader
from mctnet_model import MCTNet, MCTNetConfig
from train_mctnet import (
    build_dataloaders,
    classification_metrics,
    confusion_matrix_from_predictions,
    set_seed,
    train_model,
)

print('Imports projet OK.')


In [ ]:
# -----------------------------
# Etape 1: construire les datasets .npz/.json depuis les CSV GEE
# -----------------------------

for csv_path in CSV_FILES:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV introuvable: {csv_path}')

    bundle, metadata = build_dataset_bundle(
        csv_path=csv_path,
        normalize_reflectance=True,
        reflectance_scale=10000.0,
        split_seed=SEED,
    )

    state_slug = metadata['state_name'].lower().replace(' ', '_')
    output_npz = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.npz'
    output_json = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.json'
    save_bundle(bundle, metadata, output_npz, output_json)

    print(f'[{metadata["state_name"]}] dataset ecrit: {output_npz}')
    print(f'[{metadata["state_name"]}] metadonnees: {output_json}')
    print(json.dumps(metadata['split_counts'], indent=2))
    print('-' * 80)


In [ ]:
# -----------------------------
# Etape 2: valider les dimensions des datasets
# -----------------------------

dataset_files = sorted(PROCESSED_DIR.glob('*_mctnet_dataset.npz'))

for npz_path in dataset_files:
    state_slug = npz_path.stem.replace('_mctnet_dataset', '')
    json_path = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.json'
    bundle = load_npz_bundle(npz_path)
    metadata = json.loads(json_path.read_text(encoding='utf-8'))

    print('=' * 80)
    print('State:', metadata['state_name'])
    print('Classes:', metadata['class_name_to_index'])

    for split in ['train', 'val', 'test']:
        validate_split_shapes(bundle, split)
        validate_dataloader(bundle, split, batch_size=32)

print('=' * 80)
print('Validation des dimensions terminee.')


In [ ]:
# -----------------------------
# Etape 3: choisir le dataset a entrainer
# -----------------------------

dataset_npz = PROCESSED_DIR / f'{STATE_TO_TRAIN}_mctnet_dataset.npz'
metadata_json = PROCESSED_DIR / f'{STATE_TO_TRAIN}_mctnet_dataset.json'

if not dataset_npz.exists():
    raise FileNotFoundError(f'Dataset introuvable: {dataset_npz}')
if not metadata_json.exists():
    raise FileNotFoundError(f'Metadonnees introuvables: {metadata_json}')

bundle = load_npz_bundle(dataset_npz)
metadata = json.loads(metadata_json.read_text(encoding='utf-8'))

print('Dataset selectionne:', dataset_npz)
print('State:', metadata['state_name'])
print('Classes:', metadata['class_name_to_index'])


In [ ]:
# -----------------------------
# Etape 4: entrainer MCTNet
# -----------------------------

set_seed(SEED)

args = SimpleNamespace(
    dataset_npz=str(dataset_npz),
    metadata_json=str(metadata_json),
    output_dir=str(RUN_DIR),
    checkpoint_path=str(RUN_DIR / 'best_mctnet.pt'),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    n_stages=3,
    n_heads=5,
    kernel_size=3,
    seed=SEED,
    num_workers=0,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    cpu=False,
    no_alpe=False,
    no_mask=False,
    no_cnn=False,
    no_trans=False,
)

model, metrics_payload = train_model(args=args, bundle=bundle, metadata=metadata)

metrics_path = RUN_DIR / 'metrics.json'
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

print('Best validation metrics:')
print(json.dumps(metrics_payload['best_val'], indent=2))
print('Test metrics at best validation epoch:')
print(json.dumps(metrics_payload['test_at_best_val'], indent=2))
print('Checkpoint ecrit dans:', RUN_DIR / 'best_mctnet.pt')
print('Metrics ecrites dans:', metrics_path)


In [ ]:
# -----------------------------
# Etape 5: charger le meilleur checkpoint et calculer la matrice de confusion
# -----------------------------

def evaluate_best_checkpoint_and_confusion_matrix(bundle, metadata, checkpoint_path, batch_size=64, force_cpu=False):
    device = torch.device('cuda' if torch.cuda.is_available() and not force_cpu else 'cpu')
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model_config = MCTNetConfig(**checkpoint['model_config'])
    model = MCTNet(model_config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    loaders = build_dataloaders(bundle=bundle, batch_size=batch_size, num_workers=0)

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in loaders['test']:
            x = batch['x'].to(device)
            valid_mask = batch['valid_mask'].to(device)
            y = batch['y'].to(device)

            logits = model(x=x, valid_mask=valid_mask)
            preds = logits.argmax(dim=1)

            y_true.append(y.cpu().numpy())
            y_pred.append(preds.cpu().numpy())

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)

    num_classes = len(metadata['class_name_to_index'])
    cm = confusion_matrix_from_predictions(y_true=y_true, y_pred=y_pred, num_classes=num_classes)
    metrics = classification_metrics(y_true=y_true, y_pred=y_pred, num_classes=num_classes)

    class_names = [None] * num_classes
    for class_name, class_index in metadata['class_name_to_index'].items():
        class_names[int(class_index)] = class_name

    return cm, metrics, class_names


def plot_confusion_matrix(cm, class_names, normalize=True, figsize=(10, 8)):
    matrix = cm.astype(np.float64)
    if normalize:
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums != 0)

    fig, ax = plt.subplots(figsize=figsize)
    image = ax.imshow(matrix, cmap='Blues')
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title('Normalized confusion matrix' if normalize else 'Confusion matrix')

    threshold = matrix.max() / 2.0 if matrix.size > 0 else 0.0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            text = f'{value:.2f}' if normalize else str(int(value))
            ax.text(
                j,
                i,
                text,
                ha='center',
                va='center',
                color='white' if value > threshold else 'black',
                fontsize=9,
            )

    fig.tight_layout()
    plt.show()


In [ ]:
checkpoint_path = RUN_DIR / 'best_mctnet.pt'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint introuvable: {checkpoint_path}')

cm, test_metrics, class_names = evaluate_best_checkpoint_and_confusion_matrix(
    bundle=bundle,
    metadata=metadata,
    checkpoint_path=checkpoint_path,
    batch_size=64,
    force_cpu=False,
)

print('Test metrics from best checkpoint:')
print(json.dumps(test_metrics, indent=2))
print('Class names:', class_names)

plot_confusion_matrix(cm, class_names, normalize=True, figsize=(10, 8))


In [ ]:
# Ablations rapides du papier si besoin:
# args.no_alpe = True
# args.no_mask = True
# args.no_cnn = True
# args.no_trans = True
# Puis relancer la cellule d'entrainement.
